# Ray tracing and simplicial sheet fields

This compact example initializes a hydro grid and four-ray beam, traces the rays through a linear density gradient, and builds the two sheet-resolved tetrahedral fields.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyGATH.fields import (
    interpolate_simplicial_fields_batched,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_simplicial_mesh

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/tetrahedral_linear_gradient.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    field = simplicialise_sheet_fields(
        trace.sheet_fields,
        dimension=3,
        fields=("phase_length", "path_length", "inverse_brems_deposition"),
    )
print(
    f"{field.mesh.nsimplices:,} tetrahedra per sheet; terminated={bool(trace.terminated)}"
)

In [ ]:
figure = plt.figure(figsize=(11, 5))
for sheet in range(field.mesh.nsheets):
    axis = figure.add_subplot(1, field.mesh.nsheets, sheet + 1, projection="3d")
    plot_simplicial_mesh(field, sheet_index=sheet, ax=axis)
    axis.set_title(f"Sheet {sheet + 1}")
figure.tight_layout()

## Phase-length slices

The two panels sample the affine sheet fields on the Cartesian plane $z=0$. Values outside each sheet are masked.

In [ ]:
nx, ny = 240, 240
x = np.linspace(float(grid.xb[0]), float(grid.xb[-1]), nx)
y = np.linspace(float(grid.yb[0]), float(grid.yb[-1]), ny)
xx, yy = np.meshgrid(x, y, indexing="xy")
slice_points = np.column_stack(
    (xx.ravel(), yy.ravel(), np.zeros(xx.size, dtype=np.float64))
)
phase_slice = interpolate_simplicial_fields_batched(
    field, slice_points, point_batch_size=4096
)
phase_values = np.asarray(phase_slice.values)[
    0, :, :, field.selection.phase_length
].reshape((field.mesh.nsheets, ny, nx))
inside = np.asarray(phase_slice.inside)[0].reshape((field.mesh.nsheets, ny, nx))
masked_phase = np.ma.masked_where(~inside, phase_values)
valid_phase = np.concatenate([values.compressed() for values in masked_phase])
phase_limits = (valid_phase.min(), valid_phase.max())

figure, axes = plt.subplots(
    1, field.mesh.nsheets, figsize=(12, 4.5), sharex=True, sharey=True
)
for sheet, axis in enumerate(np.atleast_1d(axes)):
    image = axis.pcolormesh(
        x * 1.0e3,
        y * 1.0e3,
        masked_phase[sheet],
        shading="auto",
        vmin=phase_limits[0],
        vmax=phase_limits[1],
    )
    axis.set_title(f"Beam 1, sheet {sheet + 1}")
    axis.set_xlabel("x [mm]")
    axis.set_aspect("equal")
axes[0].set_ylabel("y [mm]")
figure.colorbar(image, ax=axes, label="phase length [m]", shrink=0.9)
figure.suptitle(r"Phase-length slices at $z=0$")